# Notebook 01: Data Preprocessing
## Wearable-Enhanced Insurance Underwriting — NHANES 2003-2004

**Author:** Lubaba Hassan | 22097014D | PolyU Data Science and Analytics

This notebook:
1. Loads all NHANES 2003-2004 files
2. Merges them on the common participant key (SEQN)
3. Constructs a **three-class underwriting risk label** using the D'Agostino (2008) Cox model with ATP III thresholds
4. Engineers features for all three scenarios
5. Saves three clean CSVs ready for modelling

### Label Construction — Hybrid Approach

The 10-year cardiovascular disease risk is computed using the **sex-specific Cox proportional hazards
model** from D'Agostino et al. (2008), which retains the full continuous relationship between risk
factors and CVD probability. The resulting continuous risk score is then classified into **three
clinically meaningful underwriting tiers** following the ATP III guidelines (NCEP, 2001):

| Class | Label | FRS Threshold | Underwriting Action |
|-------|-------|--------------|---------------------|
| 2 | **High Risk** | FRS ≥ 20% | Premium loading or decline; statin treatment for all |
| 1 | **Intermediate Risk** | FRS 10–19% | Specialist review; conditional treatment |
| 0 | **Low Risk** | FRS < 10% | Standard terms; statins generally not indicated |

This hybrid approach provides **more precise risk estimation** than the point-based worksheet
(which bins continuous variables into coarse categories), while applying the **same clinically
validated tier boundaries** used in practice.

**References:**
- D'Agostino, R.B. et al. (2008). General cardiovascular risk profile for use in primary care. *Circulation*, 117(6), 743-753.
- National Cholesterol Education Program, ATP III (2001). *JAMA*, 285(19):2486-97.
- Canadian Cardiovascular Society FRS Worksheet (2017).

**Feature Scenarios:**
- Scenario A: Traditional clinical features only
- Scenario B: Traditional + wearable-derived activity features
- Scenario C: Wearable-derived features only

## 1. Imports and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

DATA_PATH = '../data/processed/'
OUTPUT_PATH = '../data/processed/'
FIGURES_PATH = '../reports/figures/'
os.makedirs(FIGURES_PATH, exist_ok=True)

print('All imports successful')
print(f'Data path: {os.path.abspath(DATA_PATH)}')

## 2. Load All NHANES Files

In [ ]:
print('Loading NHANES files...')

demo   = pd.read_csv(DATA_PATH + 'DEMO_C.csv')
bmx    = pd.read_csv(DATA_PATH + 'BMX_C.csv')
bpx    = pd.read_csv(DATA_PATH + 'BPX_C.csv')
smq    = pd.read_csv(DATA_PATH + 'SMQ_C.csv')
paq    = pd.read_csv(DATA_PATH + 'PAQ_C.csv')
paqiaf = pd.read_csv(DATA_PATH + 'PAQIAF_C.csv')
mcq    = pd.read_csv(DATA_PATH + 'MCQ_C.csv')
diq    = pd.read_csv(DATA_PATH + 'DIQ_C.csv')
bpq    = pd.read_csv(DATA_PATH + 'BPQ_C.csv')
hsq    = pd.read_csv(DATA_PATH + 'HSQ_C.csv')
l13    = pd.read_csv(DATA_PATH + 'L13_C.csv')
l13am  = pd.read_csv(DATA_PATH + 'L13AM_C.csv')

print('Files loaded successfully:')
files = {
    'DEMO': demo, 'BMX': bmx, 'BPX': bpx, 'SMQ': smq,
    'PAQ': paq, 'PAQIAF': paqiaf, 'MCQ': mcq, 'DIQ': diq,
    'BPQ': bpq, 'HSQ': hsq, 'L13': l13, 'L13AM': l13am
}
for name, df in files.items():
    print(f'  {name}: {df.shape[0]:,} rows, {df.shape[1]} columns')

## 3. Engineer Wearable Features from PAQIAF

PAQIAF contains individual activity records with MET values per participant.
We aggregate to one row per participant to create wearable-proxy features.

In [ ]:
print('PAQIAF columns:', paqiaf.columns.tolist())
print('\nSample:')
print(paqiaf.head())
print('\nPADMETS stats:')
print(paqiaf['PADMETS'].describe())
print('\nPADLEVEL distribution:')
print(paqiaf['PADLEVEL'].value_counts())

In [ ]:
wearable_features = paqiaf.groupby('SEQN').agg(
    total_activities    = ('PADACTIV', 'count'),
    mean_met            = ('PADMETS', 'mean'),
    total_met_hours     = ('PADMETS', 'sum'),
    moderate_count      = ('PADLEVEL', lambda x: (x == 1.0).sum()),
    vigorous_count      = ('PADLEVEL', lambda x: (x == 2.0).sum()),
).reset_index()

wearable_features['vigorous_ratio'] = (
    wearable_features['vigorous_count'] / 
    wearable_features['total_activities'].replace(0, np.nan)
)

wearable_features['activity_category'] = pd.cut(
    wearable_features['total_met_hours'],
    bins=[0, 7.5, 15, float('inf')],
    labels=[0, 1, 2],
    include_lowest=True
).astype(float)

print(f'Wearable features created for {len(wearable_features):,} participants')
print('\nWearable feature summary:')
print(wearable_features.describe())

## 4. Build Traditional Features

In [ ]:
# ── DEMOGRAPHICS ──
demo_clean = demo[['SEQN', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'INDFMINC']].copy()
demo_clean.columns = ['SEQN', 'gender', 'age', 'ethnicity', 'income_cat']
demo_clean['male'] = (demo_clean['gender'] == 1).astype(int)
print('Demographics loaded:', demo_clean.shape)
print('Age range:', demo_clean['age'].min(), '-', demo_clean['age'].max())
print('Gender distribution:')
print(demo_clean['male'].value_counts())

In [ ]:
# ── BODY MEASUREMENTS ──
bmx_clean = bmx[['SEQN', 'BMXBMI', 'BMXWT', 'BMXHT', 'BMXWAIST']].copy()
bmx_clean.columns = ['SEQN', 'bmi', 'weight_kg', 'height_cm', 'waist_cm']
bmx_clean['bmi_cat'] = pd.cut(
    bmx_clean['bmi'], bins=[0, 18.5, 25, 30, float('inf')],
    labels=[0, 1, 2, 3], include_lowest=True
).astype(float)
print('Body measurements loaded:', bmx_clean.shape)
print('BMI stats:')
print(bmx_clean['bmi'].describe())

In [ ]:
# ── BLOOD PRESSURE (MEASURED) ──
bpx_clean = bpx[['SEQN', 'BPXSY1', 'BPXSY2', 'BPXSY3',
                  'BPXDI1', 'BPXDI2', 'BPXDI3', 'BPXPLS']].copy()
bpx_clean['systolic_bp'] = bpx_clean[['BPXSY1', 'BPXSY2', 'BPXSY3']].mean(axis=1)
bpx_clean['diastolic_bp'] = bpx_clean[['BPXDI1', 'BPXDI2', 'BPXDI3']].mean(axis=1)
bpx_clean['pulse_rate'] = bpx_clean['BPXPLS']
bpx_clean['bp_hypertension_measured'] = (
    (bpx_clean['systolic_bp'] >= 140) | (bpx_clean['diastolic_bp'] >= 90)
).astype(int)
bpx_clean = bpx_clean[['SEQN', 'systolic_bp', 'diastolic_bp',
                         'pulse_rate', 'bp_hypertension_measured']]
print('Blood pressure loaded:', bpx_clean.shape)
print('BP stats:')
print(bpx_clean[['systolic_bp', 'diastolic_bp']].describe())

In [ ]:
# ── SMOKING ──
smq_clean = smq[['SEQN', 'SMQ020']].copy()
smq_clean.columns = ['SEQN', 'smoker_raw']
smq_clean['ever_smoker'] = smq_clean['smoker_raw'].map({1.0: 1, 2.0: 0})
smq_clean = smq_clean[['SEQN', 'ever_smoker']]
print('Smoking loaded:', smq_clean.shape)
print('Smoker distribution:')
print(smq_clean['ever_smoker'].value_counts())

In [ ]:
# ── CHOLESTEROL (LAB VALUES) ──
l13_clean = l13[['SEQN', 'LBXTC', 'LBXHDD']].copy()
l13_clean.columns = ['SEQN', 'total_cholesterol', 'hdl_cholesterol']
l13am_clean = l13am[['SEQN', 'LBDLDL', 'LBXTR']].copy()
l13am_clean.columns = ['SEQN', 'ldl_cholesterol', 'triglycerides']
l13_clean['high_total_chol'] = (l13_clean['total_cholesterol'] > 200).astype(int)
l13_clean['low_hdl'] = (l13_clean['hdl_cholesterol'] < 40).astype(int)
print('Total cholesterol/HDL loaded:', l13_clean.shape)
print('LDL/Triglycerides loaded:', l13am_clean.shape)
print('High cholesterol rate:', l13_clean['high_total_chol'].mean().round(3))

In [ ]:
# ── DIABETES ──
# DIQ010: 1=Yes, 2=No, 3=Borderline, 7=Refused, 9=Don't know
diq_label = diq[['SEQN', 'DIQ010']].copy()
diq_label.columns = ['SEQN', 'diabetes_raw']
diq_label['diabetes'] = diq_label['diabetes_raw'].map({1.0: 1, 2.0: 0, 3.0: 0})
diq_label = diq_label[['SEQN', 'diabetes']]
print('Diabetes loaded:', diq_label.shape)
print('Diabetes distribution:')
print(diq_label['diabetes'].value_counts())

In [ ]:
# ── PHYSICAL ACTIVITY QUESTIONNAIRE ──
paq_clean = paq[['SEQN', 'PAQ180', 'PAD320']].copy()
paq_clean.columns = ['SEQN', 'daily_activity_level', 'days_walked']
paq_clean['daily_activity_level'] = paq_clean['daily_activity_level'].where(
    paq_clean['daily_activity_level'].isin([1, 2, 3, 4]))
print('Physical activity questionnaire loaded:', paq_clean.shape)
print('Daily activity distribution:')
print(paq_clean['daily_activity_level'].value_counts())

In [ ]:
# ── HEALTH STATUS ──
hsq_clean = hsq[['SEQN', 'HSD010']].copy()
hsq_clean.columns = ['SEQN', 'self_rated_health']
hsq_clean['self_rated_health'] = hsq_clean['self_rated_health'].where(
    hsq_clean['self_rated_health'].isin([1, 2, 3, 4, 5]))
print('Health status loaded:', hsq_clean.shape)

## 5. Construct the Three-Class Underwriting Risk Label

### Methodology: Hybrid Approach

**Step 1 — Risk Computation:** The 10-year CVD risk is computed using the sex-specific
Cox proportional hazards model from D'Agostino et al. (2008). The model uses
log-transformed continuous covariates with sex-specific regression coefficients.

**Step 2 — Risk Classification:** The continuous risk score is classified into three tiers
following the ATP III clinical guidelines (NCEP, 2001):
- **High Risk (2):** FRS ≥ 20%
- **Intermediate Risk (1):** FRS 10–19%
- **Low Risk (0):** FRS < 10%

In [ ]:
# ── BP treatment status for FRS ──
bpq_treatment = bpq[['SEQN', 'BPQ050A']].copy()
bpq_treatment.columns = ['SEQN', 'bp_treatment_raw']
bpq_treatment['on_bp_treatment'] = (bpq_treatment['bp_treatment_raw'] == 1.0).astype(int)
bpq_treatment = bpq_treatment[['SEQN', 'on_bp_treatment']]

bpq_extra = bpq[['SEQN', 'BPQ020', 'BPQ080']].copy()
bpq_extra.columns = ['SEQN', 'hypertension_told', 'high_chol_told']
bpq_extra['hypertension_told'] = (bpq_extra['hypertension_told'] == 1.0).astype(int)
bpq_extra['high_chol_told'] = (bpq_extra['high_chol_told'] == 1.0).astype(int)
print('BP treatment status:')
print(bpq_treatment['on_bp_treatment'].value_counts())

In [ ]:
# ── Assemble Framingham input dataframe ──
frs_df = demo_clean[['SEQN', 'age', 'male']].copy()
frs_df = frs_df.merge(l13_clean[['SEQN', 'total_cholesterol', 'hdl_cholesterol']],
                      on='SEQN', how='left')
frs_df = frs_df.merge(bpx_clean[['SEQN', 'systolic_bp']], on='SEQN', how='left')
frs_df = frs_df.merge(bpq_treatment, on='SEQN', how='left')
frs_df = frs_df.merge(diq_label, on='SEQN', how='left')
frs_df = frs_df.merge(smq_clean, on='SEQN', how='left')

frs_df['on_bp_treatment'] = frs_df['on_bp_treatment'].fillna(0)
frs_df['diabetes']        = frs_df['diabetes'].fillna(0)
frs_df['ever_smoker']     = frs_df['ever_smoker'].fillna(0)

# Filter to adults 30+ with required lab data
frs_eligible = frs_df[
    (frs_df['age'] >= 30) &
    frs_df['total_cholesterol'].notna() &
    frs_df['hdl_cholesterol'].notna() &
    frs_df['systolic_bp'].notna()
].copy()

print(f'FRS-eligible participants (age >= 30 with lab data): {len(frs_eligible):,}')
print(f'Excluded (too young or missing labs): {len(frs_df) - len(frs_eligible):,}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# FRAMINGHAM RISK SCORE — VECTORIZED COMPUTATION
# ══════════════════════════════════════════════════════════════════
# D'Agostino et al. (2008) Circulation 117(6):743-753, Table 2
#
# risk = 1 - S0^exp(LP - mean_coeff)
#
# IMPORTANT: The women's model uses the simplified specification
# WITHOUT the ln(age)^2 term. The age-squared term belongs to a
# different model specification with different baseline survival
# and mean coefficient values. Mixing them causes massive
# overestimation of women's risk.
# ══════════════════════════════════════════════════════════════════

print('Computing D\'Agostino (2008) Framingham Risk Scores (vectorized)...\n')

age_v  = np.clip(frs_eligible['age'].values.astype(float), 30.0, 74.0)
tc_v   = frs_eligible['total_cholesterol'].values.astype(float)
hdl_v  = frs_eligible['hdl_cholesterol'].values.astype(float)
sbp_v  = frs_eligible['systolic_bp'].values.astype(float)
bptx_v = frs_eligible['on_bp_treatment'].values.astype(float)
dm_v   = frs_eligible['diabetes'].values.astype(float)
smk_v  = frs_eligible['ever_smoker'].values.astype(float)
male_v = frs_eligible['male'].values.astype(int)

ln_age = np.log(age_v)
ln_tc  = np.log(tc_v)
ln_hdl = np.log(hdl_v)
ln_sbp = np.log(sbp_v)

ln_sbp_tx   = np.where(bptx_v == 1, ln_sbp, 0.0)
ln_sbp_notx = np.where(bptx_v == 0, ln_sbp, 0.0)

# ── Men: Table 2 coefficients ──
lp_men = (3.06117 * ln_age +
          1.12370 * ln_tc  +
         -0.93263 * ln_hdl +
          1.93303 * ln_sbp_tx +
          1.99881 * ln_sbp_notx +
          0.65451 * dm_v +
          0.57367 * smk_v)
risk_men = 1.0 - (0.88936 ** np.exp(lp_men - 23.9802))

# ── Women: Simplified model (NO age-squared term) ──
lp_women = (2.32888 * ln_age +
            0.70833 * ln_tc +
           -0.87328 * ln_hdl +
            2.76157 * ln_sbp_tx +
            2.82263 * ln_sbp_notx +
            0.52873 * smk_v +
            0.52873 * dm_v)
risk_women = 1.0 - (0.95012 ** np.exp(lp_women - 26.1931))

risk = np.where(male_v == 1, risk_men, risk_women)
risk = np.clip(risk, 0.0, 1.0)

frs_eligible['framingham_score'] = risk
frs_eligible['framingham_pct']   = risk * 100
frs_eligible = frs_eligible.dropna(subset=['framingham_score'])

print(f'FRS computed successfully: {len(frs_eligible):,} participants')
print(f'\nContinuous risk score distribution (%):')
print(frs_eligible['framingham_pct'].describe().round(2))
print(f'\nMedian risk % by gender (0=Female, 1=Male):')
print(frs_eligible.groupby('male')['framingham_pct'].median().round(2))

In [ ]:
# ── Apply ATP III three-class thresholds ──
def classify_frs_risk(risk_pct):
    if risk_pct >= 20.0:
        return 2, 'High'
    elif risk_pct >= 10.0:
        return 1, 'Intermediate'
    else:
        return 0, 'Low'

classes, labels = zip(*frs_eligible['framingham_pct'].apply(classify_frs_risk))
frs_eligible['risk_class'] = list(classes)
frs_eligible['risk_label'] = list(labels)
frs_eligible['risk_class'] = frs_eligible['risk_class'].astype(int)

print('THREE-CLASS LABEL DISTRIBUTION (ATP III Thresholds)')
print('=' * 55)
for label in ['Low', 'Intermediate', 'High']:
    count = (frs_eligible['risk_label'] == label).sum()
    pct = count / len(frs_eligible) * 100
    cls = {'Low': 0, 'Intermediate': 1, 'High': 2}[label]
    print(f'  Class {cls} - {label:15s}: {count:>5,}  ({pct:>5.1f}%)')
print(f'  Total:              {len(frs_eligible):>5,}')

print(f'\nRisk % statistics by class:')
print(frs_eligible.groupby('risk_label')['framingham_pct'].describe().round(2))

In [ ]:
# Prepare label dataframe for merge
label_df = frs_eligible[['SEQN', 'framingham_score', 'framingham_pct',
                          'risk_class', 'risk_label']].copy()
print(f'Label dataframe ready: {len(label_df):,} participants')
print(label_df['risk_class'].value_counts().sort_index())

In [ ]:
# ── Visualise FRS distribution and three-class labels ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.hist(label_df['framingham_pct'].clip(0, 60), bins=60,
        color='#3498db', edgecolor='white', alpha=0.8)
ax.axvline(x=10, color='orange', linestyle='--', linewidth=2, label='10% (Low to Intermediate)')
ax.axvline(x=20, color='red', linestyle='--', linewidth=2, label='20% (Intermediate to High)')
ax.set_xlabel('10-Year CVD Risk (%)')
ax.set_ylabel('Count')
ax.set_title('Continuous FRS Distribution\n(D\'Agostino Cox Model)', fontweight='bold')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax = axes[1]
class_order = ['Low', 'Intermediate', 'High']
class_colors = ['#2ecc71', '#f39c12', '#e74c3c']
counts = [label_df[label_df['risk_label'] == c].shape[0] for c in class_order]
bars = ax.bar(class_order, counts, color=class_colors, edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, counts):
    pct = count / len(label_df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold', fontsize=9)
ax.set_ylabel('Count')
ax.set_title('Three-Class Underwriting Label\n(ATP III Thresholds)', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax = axes[2]
merged_gender = label_df.merge(frs_eligible[['SEQN', 'male']], on='SEQN')
for gender, color, name in [(1, '#3498db', 'Male'), (0, '#e74c3c', 'Female')]:
    pts = merged_gender[merged_gender['male'] == gender]['framingham_pct']
    ax.hist(pts, bins=40, alpha=0.6, color=color, label=name, density=True)
ax.axvline(x=10, color='orange', linestyle='--', linewidth=1.5, alpha=0.7)
ax.axvline(x=20, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
ax.set_xlabel('10-Year CVD Risk (%)')
ax.set_ylabel('Density')
ax.set_title('Risk Distribution by Gender', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.suptitle('Framingham Risk Score — Hybrid Approach\nD\'Agostino (2008) Cox Model + ATP III Three-Class Thresholds',
             fontsize=13, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig(FIGURES_PATH + 'framingham_three_class_label.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 6. Merge All Data

In [ ]:
df = demo_clean.copy()
df = df.merge(bmx_clean,   on='SEQN', how='left')
df = df.merge(bpx_clean,   on='SEQN', how='left')
df = df.merge(smq_clean,   on='SEQN', how='left')
df = df.merge(l13_clean,   on='SEQN', how='left')
df = df.merge(l13am_clean, on='SEQN', how='left')
df = df.merge(paq_clean,   on='SEQN', how='left')
df = df.merge(hsq_clean,   on='SEQN', how='left')
df = df.merge(bpq_extra,   on='SEQN', how='left')
df = df.merge(bpq_treatment, on='SEQN', how='left')
df = df.merge(wearable_features, on='SEQN', how='left')
df = df.merge(label_df, on='SEQN', how='inner')
df['has_wearable'] = (~df['total_met_hours'].isna()).astype(int)

print(f'Full merged dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'Participants with wearable data: {df["has_wearable"].sum():,} ({df["has_wearable"].mean():.1%})')
print(f'\nRisk class distribution:')
print(df['risk_label'].value_counts())

## 7. Data Cleaning

In [ ]:
print(f'Starting dataset: {df.shape[0]:,} participants (all age >= 30)')
df = df[(df['bmi'].isna()) | ((df['bmi'] >= 10) & (df['bmi'] <= 80))]
print(f'After BMI outlier removal: {df.shape[0]:,} participants')
df = df[(df['systolic_bp'].isna()) | ((df['systolic_bp'] >= 70) & (df['systolic_bp'] <= 250))]
print(f'After BP outlier removal: {df.shape[0]:,} participants')
df = df[df['age'] <= 85]
print(f'After age cap (85): {df.shape[0]:,} participants')
print(f'\nFinal dataset size: {df.shape[0]:,} participants')
print(f'\nThree-class label distribution:')
for label in ['Low', 'Intermediate', 'High']:
    count = (df['risk_label'] == label).sum()
    pct = count / len(df) * 100
    print(f'  {label:15s}: {count:>5,} ({pct:.1f}%)')

## 8. Missing Value Analysis

In [ ]:
key_cols = [
    'age', 'male', 'bmi', 'systolic_bp', 'diastolic_bp',
    'ever_smoker', 'total_cholesterol', 'hdl_cholesterol',
    'ldl_cholesterol', 'triglycerides', 'self_rated_health',
    'daily_activity_level', 'high_chol_told',
    'total_met_hours', 'mean_met', 'vigorous_ratio',
    'moderate_count', 'vigorous_count', 'activity_category',
    'risk_class'
]
missing_summary = pd.DataFrame({
    'missing_count': df[key_cols].isnull().sum(),
    'missing_pct': (df[key_cols].isnull().sum() / len(df) * 100).round(1)
})
print('Missing value summary:')
print(missing_summary.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
missing_pct = (df[key_cols].isnull().sum() / len(df) * 100).sort_values(ascending=True)
colors = ['#e74c3c' if x > 40 else '#f39c12' if x > 20 else '#2ecc71' for x in missing_pct]
ax.barh(missing_pct.index, missing_pct.values, color=colors)
ax.axvline(x=20, color='orange', linestyle='--', alpha=0.7, label='20% threshold')
ax.axvline(x=40, color='red', linestyle='--', alpha=0.7, label='40% threshold')
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Values by Feature')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_PATH + 'missing_values.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 9. Add Age Bands and Fairness Subgroups

In [ ]:
df['age_band'] = pd.cut(df['age'], bins=[30, 45, 60, 75, 85],
                         labels=['30-45', '46-60', '61-75', '76-85'], include_lowest=True)
print('Age band distribution:')
print(df['age_band'].value_counts().sort_index())
print('\nRisk class distribution by age band (%):')
ct = pd.crosstab(df['age_band'], df['risk_label'], normalize='index') * 100
ct = ct[['Low', 'Intermediate', 'High']]
print(ct.round(1))
print('\nRisk class distribution by gender (0=Female, 1=Male) (%):')
ct2 = pd.crosstab(df['male'], df['risk_label'], normalize='index') * 100
ct2 = ct2[['Low', 'Intermediate', 'High']]
print(ct2.round(1))

## 10. Build Three Scenario Datasets

In [ ]:
TRADITIONAL_FEATURES = [
    'age', 'male', 'bmi', 'waist_cm',
    'systolic_bp', 'diastolic_bp',
    'total_cholesterol', 'hdl_cholesterol',
    'ever_smoker', 'self_rated_health',
    'daily_activity_level', 'high_chol_told',
    'bp_hypertension_measured'
]
WEARABLE_FEATURES = [
    'total_met_hours', 'mean_met', 'total_activities',
    'moderate_count', 'vigorous_count', 'vigorous_ratio',
    'activity_category'
]
META_COLS = ['SEQN', 'age_band', 'ethnicity', 'income_cat',
             'risk_class', 'risk_label', 'framingham_pct']
print('Traditional features:', len(TRADITIONAL_FEATURES))
print('Wearable features:', len(WEARABLE_FEATURES))

In [ ]:
from sklearn.impute import SimpleImputer

def prepare_scenario(df, feature_cols, label_col='risk_class',
                     meta_cols=None, strategy='median'):
    if meta_cols is None:
        meta_cols = ['SEQN', 'risk_class', 'risk_label']
    cols = list(set(meta_cols + feature_cols + [label_col]))
    scenario_df = df[cols].copy()
    scenario_df = scenario_df.dropna(subset=[label_col])
    numeric_features = [f for f in feature_cols
                        if scenario_df[f].dtype in ['float64', 'int64']]
    imputer = SimpleImputer(strategy=strategy)
    scenario_df[numeric_features] = imputer.fit_transform(scenario_df[numeric_features])
    print(f'  Shape: {scenario_df.shape}')
    print(f'  Label distribution:')
    for cls in [0, 1, 2]:
        count = (scenario_df[label_col] == cls).sum()
        pct = count / len(scenario_df) * 100
        name = {0: 'Low', 1: 'Intermediate', 2: 'High'}[cls]
        print(f'    {name} ({cls}): {count:,} ({pct:.1f}%)')
    print(f'  Missing values remaining: {scenario_df[feature_cols].isnull().sum().sum()}')
    return scenario_df

In [ ]:
print('Preparing Scenario A (Traditional features only)...')
scenario_a = prepare_scenario(df, feature_cols=TRADITIONAL_FEATURES, meta_cols=META_COLS)
scenario_a.to_csv(OUTPUT_PATH + 'scenario_a.csv', index=False)
print('  Saved: scenario_a.csv')

In [ ]:
print('Preparing Scenario B (Traditional + Wearable features)...')
df_b = df[df['has_wearable'] == 1].copy()
print(f'  Participants with wearable data: {len(df_b):,}')
scenario_b = prepare_scenario(df_b, feature_cols=TRADITIONAL_FEATURES + WEARABLE_FEATURES, meta_cols=META_COLS)
scenario_b.to_csv(OUTPUT_PATH + 'scenario_b.csv', index=False)
print('  Saved: scenario_b.csv')

In [ ]:
print('Preparing Scenario C (Wearable features only)...')
scenario_c = prepare_scenario(df_b, feature_cols=WEARABLE_FEATURES, meta_cols=META_COLS)
scenario_c.to_csv(OUTPUT_PATH + 'scenario_c.csv', index=False)
print('  Saved: scenario_c.csv')

## 11. Summary Statistics

In [ ]:
print('=' * 65)
print('PREPROCESSING COMPLETE')
print('=' * 65)
print(f'Total NHANES participants loaded:       {demo.shape[0]:,}')
print(f'FRS-eligible (age >= 30 + labs):         {len(label_df):,}')
print(f'After cleaning:                         {df.shape[0]:,}')
print()
print('LABEL CONSTRUCTION:')
print('  Risk computation:  D\'Agostino (2008) Cox model')
print('  Risk thresholds:   ATP III guidelines (NCEP, 2001)')
print('    Low (0):           FRS < 10%')
print('    Intermediate (1):  FRS 10-19%')
print('    High (2):          FRS >= 20%')
print()
for name, scen, n_feat in [('A (Traditional)', scenario_a, len(TRADITIONAL_FEATURES)),
                            ('B (Traditional + Wearable)', scenario_b, len(TRADITIONAL_FEATURES) + len(WEARABLE_FEATURES)),
                            ('C (Wearable only)', scenario_c, len(WEARABLE_FEATURES))]:
    print(f'Scenario {name}:')
    print(f'  Participants: {len(scen):,}  |  Features: {n_feat}')
    for cls, cname in [(0, 'Low'), (1, 'Intermediate'), (2, 'High')]:
        pct = (scen['risk_class'] == cls).mean() * 100
        print(f'    {cname}: {pct:.1f}%')
    print()
print('Files saved: scenario_a.csv, scenario_b.csv, scenario_c.csv')

In [ ]:
# Label distribution plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
scenarios = [
    (scenario_a, 'Scenario A\n(Traditional)'),
    (scenario_b, 'Scenario B\n(Traditional + Wearable)'),
    (scenario_c, 'Scenario C\n(Wearable Only)')
]
class_names = ['Low\n(FRS<10%)', 'Intermediate\n(FRS 10-19%)', 'High\n(FRS>=20%)']
class_colors = ['#2ecc71', '#f39c12', '#e74c3c']
for ax, (scen_df, title) in zip(axes, scenarios):
    counts = [int((scen_df['risk_class'] == c).sum()) for c in [0, 1, 2]]
    bars = ax.bar(class_names, counts, color=class_colors, edgecolor='white', linewidth=1.5)
    for bar, count in zip(bars, counts):
        pct = count / sum(counts) * 100
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold', fontsize=8)
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_ylabel('Count')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
plt.suptitle('Three-Class Label Distribution Across Feature Scenarios\n'
             'D\'Agostino (2008) Cox Model + ATP III Thresholds',
             fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(FIGURES_PATH + 'label_distribution_three_class.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# Risk class by age band
fig, ax = plt.subplots(figsize=(10, 6))
risk_by_age = pd.crosstab(df['age_band'], df['risk_label'], normalize='index') * 100
risk_by_age = risk_by_age[['Low', 'Intermediate', 'High']]
risk_by_age.plot(kind='bar', stacked=True, ax=ax,
                 color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='white', linewidth=0.5)
ax.set_xlabel('Age Band')
ax.set_ylabel('Percentage (%)')
ax.set_title('Risk Class Distribution by Age Band\n(Validates Clinical Plausibility)', fontweight='bold')
ax.legend(title='Risk Class', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_PATH + 'risk_by_age_band.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# Wearable feature distributions by risk class
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
wearable_plot_cols = [
    ('total_met_hours', 'Total MET-Hours'), ('mean_met', 'Mean MET Value'),
    ('total_activities', 'Total Activities'), ('moderate_count', 'Moderate Activity Count'),
    ('vigorous_count', 'Vigorous Activity Count'), ('vigorous_ratio', 'Vigorous Activity Ratio')
]
df_with_wearable = df[df['has_wearable'] == 1]
for ax, (col, title) in zip(axes.flat, wearable_plot_cols):
    for cls, color, name in [(0, '#2ecc71', 'Low'), (1, '#f39c12', 'Intermediate'), (2, '#e74c3c', 'High')]:
        data = df_with_wearable[df_with_wearable['risk_class'] == cls][col].dropna()
        ax.hist(data, bins=30, alpha=0.5, color=color, label=name, density=True)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
plt.suptitle('Wearable Feature Distributions by Risk Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_PATH + 'wearable_distributions_three_class.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# Validation spot-check
print('VALIDATION: Sample FRS computations')
print('=' * 70)
sample = frs_eligible.sample(5, random_state=42)
check_cols = ['SEQN', 'age', 'male', 'total_cholesterol', 'hdl_cholesterol',
              'systolic_bp', 'on_bp_treatment', 'diabetes', 'ever_smoker',
              'framingham_pct', 'risk_label']
print(sample[check_cols].to_string(index=False))
print('\nVerification notes:')
print('  - Women\'s model uses simplified specification (no age-squared term)')
print('  - Risk % thresholds: <10% = Low, 10-19% = Intermediate, >=20% = High')
print('  - Cholesterol values are in mg/dL (NHANES native units)')

In [ ]:
print('Notebook 01 complete.')
print('Next: Run 02_eda.ipynb for exploratory data analysis')
print('Then: Run 03_modelling.ipynb to train multi-class classifiers')